# Neon isotopes: HF, HFB, and fidelity step by step

This notebook rebuilds the essential calculation behind `study_gaussian_fidelity.py` without calling its `run_study(...)` wrapper. Each cell performs one visible physics or numerical step.

We begin with $^{20}$Ne. Once that calculation is understood and converged, the last section shows how to repeat it for $^{22}$Ne and $^{24}$Ne.

## 1. What we will calculate

For one isotope we will construct three states:

1. the exact shell-model ground state $|\Psi_0\rangle$ in the fixed $(N,Z,M=0)$ sector;
2. the energy-minimized Hartree-Fock determinant $|\Phi_{\rm HF}\rangle$;
3. the energy-minimized Hartree-Fock-Bogoliubov vacuum $|\Phi_{\rm HFB}\rangle$.

We then evaluate the **intrinsic** fidelities

$$F_{\rm HF}=|\langle\Psi_0|\Phi_{\rm HF}\rangle|^2,\qquad F_{\rm HFB}=|\langle\Psi_0|\Phi_{\rm HFB}\rangle|^2.$$

These are direct overlaps. HFB may contain components with the wrong particle numbers, so its intrinsic fidelity includes both the quality of the wavefunction and the probability of being in the requested sector. We will separate those effects later with particle-number projection.

In [ ]:
from pathlib import Path
import sys
import numpy as np

# Locate the repository even if Jupyter starts in a subdirectory.
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook from inside the repository")
    ROOT = ROOT.parent

for directory in (ROOT / "src" / "NSMFermions",):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from nuclear_workflow import (
    build_fermionic_hamiltonian,
    load_nuclear_interaction,
    nucleus_from_mass,
)
from hfb import HFBHamiltonian, HFBState, solve_hfb
from gaussian_fidelity import maximize_best_gaussian_fidelity
from number_projection import exact_ground_state, project_particle_numbers

print("Repository:", ROOT)

The three workflow helpers above only handle repository data plumbing: `load_nuclear_interaction` reads USDB, `nucleus_from_mass` converts a mass number into valence-particle counts, and `build_fermionic_hamiltonian` assembles a fixed-sector matrix. The HF, HFB, exact diagonalization, projection, and overlap calculations remain explicit below.

## 2. Choose one nucleus and numerical effort

With an inert $^{16}$O core, every Neon isotope has two valence protons and $A-18$ valence neutrons. Start in learning mode. Increase `STARTS` and `MAXITER` only after every object and diagnostic is clear.

In [ ]:
MASS = 20
STARTS = 3          # includes the spherical seed plus two random starts
MAXITER = 100       # use about 500 for production
OVERLAP_STARTS = 3  # use 8 or more for the non-convex overlap search
OVERLAP_MAXITER = 500
SEED = 8
REAL_BOGOLIUBOV = True

print(f"Selected isotope: {MASS}Ne")

## 3. Load the USDB interaction

`interaction[(i,j,k,l)]` contains the antisymmetrized two-body matrix elements, `eps[i]` contains the one-body energies, and `encoding[i]` stores the quantum numbers of single-particle mode $i$. USDB has 12 proton modes followed by 12 neutron modes in this repository. The first execution can take roughly twenty seconds because the coupled-basis interaction is transformed to the single-particle basis.

In [ ]:
interaction, eps, encoding, interaction_path = load_nuclear_interaction(
    "usdb", repository_root=ROOT
)
modes = len(eps)
species_modes = modes // 2
neutron_modes = list(range(species_modes, modes))
label, targets = nucleus_from_mass("usdb", MASS, species_modes)

print("Interaction file:     ", interaction_path)
print("One-body modes:       ", modes)
print("Two-body coefficients:", len(interaction))
print("Nucleus:              ", label)
print("Targets [N, Z]:       ", targets)

## 4. Construct the intrinsic Hamiltonian

`HFBHamiltonian` stores the one-body matrix $h$ and antisymmetrized two-body tensor $\bar v$. Its `energy(state)` method evaluates

$$E[\rho,\kappa]=\mathrm{Tr}(h\rho)+\frac12\bar v_{ijkl}\rho_{ki}\rho_{lj}+\frac14\bar v_{ijkl}\kappa^*_{ij}\kappa_{kl}.$$

HF is the special case $\kappa=0$ with an idempotent density matrix.

In [ ]:
intrinsic_hamiltonian = HFBHamiltonian(np.diag(eps), interaction)
print("h shape:", intrinsic_hamiltonian.h.shape)
print("v shape:", intrinsic_hamiltonian.v.shape)

## 5. Minimize the HF and HFB energies

Both approximations use the same public entry point. `method="hf"` optimizes occupied proton and neutron orbitals on number-conserving Grassmann manifolds. `method="hfb"` optimizes a Bogoliubov vacuum with average $N$ and $Z$ constrained.

For $^{20}$Ne the deterministic first start reaches a spherical stationary point near $-29.35$ MeV. The random starts are needed to find the lower deformed solution near $-37.18$ MeV. Three starts make that lesson visible; use at least eight and repeat the seed for production results.

In [ ]:
hf = solve_hfb(
    intrinsic_hamiltonian, neutron_modes, targets,
    method="hf", starts=STARTS, maxiter=MAXITER, seed=SEED + MASS,
    tolerance=1e-8,
)

hfb = solve_hfb(
    intrinsic_hamiltonian, neutron_modes, targets,
    method="hfb", starts=STARTS, maxiter=MAXITER, seed=SEED + MASS,
    tolerance=1e-8, real_bogoliubov=REAL_BOGOLIUBOV,
)

for name, result in [("HF", hf), ("HFB", hfb)]:
    print(f"{name}: E = {result.energy:.10f} MeV")
    print(f"     <N>, <Z> = {result.numbers}")
    print(f"     converged = {result.converged}")
    print(f"     stationarity error = {result.stationarity_error:.3e}")
    print(f"     ||kappa|| = {np.linalg.norm(result.state.kappa):.3e}")

Do not use a result as a production value merely because it has a low energy. Check `converged`, the number expectations, the stationarity error, and all entries in `result.attempts`. If `||kappa||` is nearly zero, the HFB minimum has collapsed to an HF-like boundary state; that can be physical rather than an optimizer failure.

## 6. Build and diagonalize the exact $(N,Z,M=0)$ problem

The exact state must be expressed in a determinant basis. Fixing $M=0$ makes the USDB diagonalization smaller. This restriction is applied only to the exact basis; the intrinsic HF and HFB optimizations above did **not** impose $M=0$.

In [ ]:
magnetic_projections = np.asarray([float(state[3]) for state in encoding])

def has_m_zero(occupied_modes):
    return np.isclose(magnetic_projections[list(occupied_modes)].sum(), 0.0)

exact_hamiltonian = build_fermionic_hamiltonian(
    interaction, eps, particles=targets, symmetries=[has_m_zero]
)
exact_energy, exact_state = exact_ground_state(exact_hamiltonian)
exact_state = np.asarray(exact_state, complex)
exact_state /= np.linalg.norm(exact_state)

print("Exact basis dimension:", len(exact_hamiltonian.occupations))
print("Nonzero matrix entries:", exact_hamiltonian.matrix.nnz)
print(f"Exact energy: {exact_energy:.10f} MeV")

## 7. Convert a Gaussian state into amplitudes and compute fidelity

`stable_normalized_occupation_amplitudes(occupations)` evaluates $\langle I|\Phi\rangle$ for every determinant $|I\rangle$ in the exact basis. HF amplitudes are determinants; paired HFB amplitudes are Pfaffians. Once both states use the same basis ordering, fidelity is just a complex inner product.

A nearly pairing-collapsed HFB solution lies at a singular boundary of the particle-vacuum Thouless chart. In that case we reconstruct the equivalent Slater determinant from the occupied natural orbitals before evaluating amplitudes.

In [ ]:
def stable_state_for_fidelity(state, particle_number):
    rho = (state.rho + state.rho.conj().T) / 2
    pairing_norm = np.linalg.norm(state.kappa)
    idempotency_error = np.linalg.norm(rho @ rho - rho)

    if pairing_norm < 1e-3 and idempotency_error < 1e-6:
        _, natural_orbitals = np.linalg.eigh(rho)
        occupied = natural_orbitals[:, -particle_number:]
        return HFBState.from_slater(occupied), "Slater boundary"
    return state, "finite Bogoliubov chart"

def intrinsic_fidelity(state, occupations, target):
    amplitudes = state.stable_normalized_occupation_amplitudes(occupations)
    return float(abs(np.vdot(target, amplitudes)) ** 2)

hfb_fidelity_state, hfb_chart = stable_state_for_fidelity(
    hfb.state, sum(targets)
)

hf_fidelity = intrinsic_fidelity(
    hf.state, exact_hamiltonian.occupations, exact_state
)
hfb_fidelity = intrinsic_fidelity(
    hfb_fidelity_state, exact_hamiltonian.occupations, exact_state
)

print(f"HF  intrinsic fidelity: {hf_fidelity:.10f}")
print(f"HFB intrinsic fidelity: {hfb_fidelity:.10f}")
print("HFB amplitude representation:", hfb_chart)

## 8. Put energy and fidelity side by side

Energy minimization and fidelity maximization are different objectives. HFB guarantees a variational space at least as large as HF, so a properly converged HFB energy should not be above the best HF energy. That does **not** guarantee that its intrinsic fidelity is larger.

In [ ]:
names = ["Exact", "HF", "HFB"]
energies = [exact_energy, hf.energy, hfb.energy]
fidelities = [1.0, hf_fidelity, hfb_fidelity]

print(f"{'state':<8} {'energy [MeV]':>16} {'fidelity':>14} {'converged':>12}")
print("-" * 54)
for name, energy, fidelity, converged in zip(
    names, energies, fidelities, [True, hf.converged, hfb.converged]
):
    print(f"{name:<8} {energy:>16.8f} {fidelity:>14.8f} {str(converged):>12}")

## 9. Maximize Gaussian overlap with the exact state

The HF and HFB states above minimize energy. We now solve a different problem:

$$\max_{\Phi\in\mathcal G}|\langle\Psi_0|\Phi\rangle|^2.$$

A finite particle-vacuum Thouless chart does not contain nonempty Slater determinants, which lie on its singular boundary. A complete best-found pure-Gaussian search therefore runs two optimizers:

1. a paired Bogoliubov search in the finite Thouless chart;
2. an HF/Slater search on the boundary.

The library returns both results and selects the one with the larger fidelity. This is still a non-convex multistart search, not a proof of the global maximum.

In [ ]:
# Use the energy-minimized states as informed initial guesses when their
# coordinate charts are numerically available.
initial_bogoliubov_parameters = None
try:
    initial_z = hfb.state.thouless_matrix
    upper = initial_z[np.triu_indices(modes, 1)]
    if REAL_BOGOLIUBOV and np.max(np.abs(upper.imag), initial=0.0) < 1e-10:
        initial_bogoliubov_parameters = upper.real
    elif not REAL_BOGOLIUBOV:
        initial_bogoliubov_parameters = np.concatenate([upper.real, upper.imag])
except ValueError:
    # A collapsed HFB state has no finite particle-vacuum Thouless matrix.
    pass

natural_occupations, natural_orbitals = np.linalg.eigh(
    (hfb.state.rho + hfb.state.rho.conj().T) / 2
)
initial_hf_orbitals = natural_orbitals[
    :, np.argsort(natural_occupations)[-sum(targets):]
]
if REAL_BOGOLIUBOV:
    initial_hf_orbitals = initial_hf_orbitals.real

closest = maximize_best_gaussian_fidelity(
    exact_hamiltonian,
    exact_state,
    bogoliubov_starts=OVERLAP_STARTS,
    hartree_fock_starts=OVERLAP_STARTS,
    seed=SEED + 1000 + MASS,
    bogoliubov_maxiter=OVERLAP_MAXITER,
    hartree_fock_maxiter=OVERLAP_MAXITER,
    gradient_tolerance=2e-6,
    real_parameters=REAL_BOGOLIUBOV,
    initial_bogoliubov_parameters=initial_bogoliubov_parameters,
    initial_hartree_fock_orbitals=initial_hf_orbitals,
)

closest_bogoliubov_energy = intrinsic_hamiltonian.energy(
    closest.bogoliubov.state
)
closest_hf_state = HFBState.from_slater(closest.hartree_fock.orbitals)
closest_hf_energy = intrinsic_hamiltonian.energy(closest_hf_state)

print("Selected family:            ", closest.family)
print(f"Best Gaussian fidelity:     {closest.fidelity:.10f}")
print(f"Bogoliubov fidelity:        {closest.bogoliubov.fidelity:.10f}")
print(f"Bogoliubov converged:       {closest.bogoliubov.converged}")
print(f"Bogoliubov energy:          {closest_bogoliubov_energy:.10f} MeV")
print(f"Bogoliubov gradient norm:   {closest.bogoliubov.gradient_norm:.3e}")
print(f"HF-boundary fidelity:       {closest.hartree_fock.fidelity:.10f}")
print(f"HF-boundary converged:      {closest.hartree_fock.converged}")
print(f"HF-boundary energy:         {closest_hf_energy:.10f} MeV")
print(f"HF-boundary gradient norm:  {closest.hartree_fock.gradient_norm:.3e}")

The maximum-overlap states need not have low energies because energy was not their objective. The HF-boundary search fixes the total particle number but permits proton-neutron orbital mixing; the Bogoliubov search may also break particle number. Therefore compare these raw intrinsic fidelities separately from the constrained energy-minimized HF/HFB results.

## 10. Optional: particle-number projection of HFB

The exact state has fixed $N$ and $Z$, while intrinsic HFB generally does not. To distinguish particle-number leakage from the shape of the state inside the desired sector, build the complete fixed-$(N,Z)$ basis, embed the $M=0$ exact state in it, and project HFB.

The projection returns:

- `sector_weight`: probability that intrinsic HFB has the requested $(N,Z)$;
- `fidelity`: overlap after selecting and renormalizing that sector;
- `energy`: particle-number-projected-after-variation energy.

For a normalized exact target, the identity $F_{\rm intrinsic}=w_{N,Z}F_{P_NP_Z}$ provides a useful numerical check. This section is optional because the complete basis is larger, especially for $^{22}$Ne and $^{24}$Ne.

In [ ]:
RUN_NUMBER_PROJECTION = False

if RUN_NUMBER_PROJECTION:
    full_hamiltonian = build_fermionic_hamiltonian(
        interaction, eps, particles=targets
    )

    full_index = {
        tuple(occupied): i
        for i, occupied in enumerate(full_hamiltonian.occupations)
    }
    exact_state_full = np.zeros(len(full_hamiltonian.occupations), complex)
    for coefficient, occupied in zip(
        exact_state, exact_hamiltonian.occupations
    ):
        exact_state_full[full_index[tuple(occupied)]] = coefficient

    projected_hfb = project_particle_numbers(
        hfb_fidelity_state, full_hamiltonian, target=exact_state_full
    )

    reconstructed = projected_hfb.sector_weight * projected_hfb.fidelity
    print("Full fixed-(N,Z) dimension:", len(full_hamiltonian.occupations))
    print(f"Target-sector weight:       {projected_hfb.sector_weight:.10f}")
    print(f"Projected HFB fidelity:     {projected_hfb.fidelity:.10f}")
    print(f"Projected HFB energy:       {projected_hfb.energy:.10f} MeV")
    print(f"weight x projected F:       {reconstructed:.10f}")
    print(f"direct intrinsic F:         {hfb_fidelity:.10f}")
else:
    print("Set RUN_NUMBER_PROJECTION=True when you are ready for this step.")

## 11. From one isotope to the Neon chain

The calculation is now explicit for one isotope. The next notebook step will package these same cells into a small `calculate_isotope(mass, ...)` function and loop over `(20, 22, 24)`. We postpone that wrapper deliberately: first inspect the `20Ne` convergence diagnostics and make sure the distinction between intrinsic and projected fidelity is clear.

When this single-isotope calculation is clear, the same explicit steps can be packaged into a small function and evaluated for the full chain without returning to a large benchmark driver.